In [ ]:
import os
os.chdir(path_to_wd)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
print(torch.cuda.is_available()) 
import scvi
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap
import sys
import scipy
import scipy.sparse as sp
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix, lil_matrix
from scipy import io
from scipy.io import mmread
import pycats
from pandas.api.types import CategoricalDtype
import hotspot
import seaborn as sns

scvi.settings.seed = 12345
plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
# color map
blue_yellow_colors = ["#352A86","#343DAE","#0262E0","#1389D2","#2DB7A3","#A5BE6A","#F8BA43","#F6DA23","#F8FA0D"]
horizon_colors = ["#000033","#000075","#0000B6","#0000F8","#2E00FF","#6100FF","#9408F7","#C729D6","#FA4AB5",
                 "#FF6A95","#FF8B74","#FFAC53","#FFCD32","#FFEE11","#FFFF60"]
solar_extra_colors = ["#3361A5","#248AF3","#14B3FF","#88CEEF","#C1D5DC","#EAD397","#FDB31A","#E42A2A","#A31D1D"]

blue_yellow = LinearSegmentedColormap.from_list('custom',blue_yellow_colors)
horizon = LinearSegmentedColormap.from_list('custom',horizon_colors)
solar_extra = LinearSegmentedColormap.from_list('custom',solar_extra_colors)

In [ ]:
colors = ["#5DADE2", "white", "#FF69B4"]  # Lighter sky blue and hot pink
custom_cmap = LinearSegmentedColormap.from_list("light_skyblue_to_hot_pink", colors)

## Load data

In [ ]:
adata_full = sc.read("Reproducibility/Data/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")

fig_dir = "Reproducibility/Results/Plots/Myeloid/"
os.makedirs(fig_dir, exist_ok = True)

In [ ]:
tmp_subset = 'Myeloid'

adata = adata_full[adata_full.obs['lineage']==tmp_subset].copy()

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_latent_df.txt"
latent_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
latent_df = latent_df.loc[adata.obs_names, : ].values 
adata.obsm["MultiVI_latent"] = latent_df

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_UMAP_df.txt"
UMAP_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
adata.obsm["X_umap"] = UMAP_df.values

sc.pp.neighbors(adata, use_rep="MultiVI_latent")

In [ ]:
from pandas.api.types import CategoricalDtype

cat_type = CategoricalDtype(categories=["Mono","MDSC-like",'TAM_TREM2',"TAM_FOLR2","cDC1","cDC2",
                                        "mregDC","preDC","pDC","Mast"], ordered=True)
adata.obs['celltype'] = adata.obs['celltype'].astype(cat_type)

## Visualization

### UMAP

In [ ]:
# Fig.S5A
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=['celltype'],
    frameon=False,
    #legend_loc="on data",
    legend_fontsize=7,
    show = False
)
plt.savefig(f"{fig_dir}FigureS5A_UMAP.pdf", bbox_inches='tight')
plt.close()

### RNA dotplot

In [ ]:
adata_RNA = adata[:, adata.var.modality == "Gene Expression"].copy()
adata_RNA.layers["counts"] = adata_RNA.X.copy()
sc.pp.normalize_total(adata_RNA)
sc.pp.log1p(adata_RNA)

adata_RNA.layers['scaled'] = sc.pp.scale(adata_RNA, copy=True).X

In [ ]:
marker_genes = ["THBS1","AQP9",'NLRP3','FCN1','IRAK3',
                'HMOX1','PPARG','SERPINE1',
                'SPP1','TREM2','GPNMB',
                'SELENOP','SLC40A1','FOLR2','LYVE1',
                'CLEC9A','XCR1',"CD226",'BTLA',
                "FCER1A","CD1C",'CLEC10A',
                'LAMP3','CCR7','CCL22','IL15',
                'RUNX3','ENTPD1','AXL','ITGAE','SPIB',
                'IRF4','BCL11A','RUNX2','SLC15A4','PBX3',
                'KIT','TPSB2','TPSAB1'
                ]

In [ ]:
# Fig.S5A
sc.pl.dotplot(adata_RNA, 
              marker_genes, 
              groupby = "celltype",
              dendrogram=False,
              cmap=custom_cmap,
              dot_max = 0.8,
              layer = 'scaled',
              use_raw=False,
              vmin = -1,
              vmax = 1,
              show = False)
plt.savefig(f"{fig_dir}FigureS5A_RNA_dotplot.pdf", bbox_inches='tight')
plt.close()

### Protein heatmap

In [ ]:
from muon import prot as pt
adt_df = adata_RNA.obsm["protein_expression"].copy()
adt_df.columns = [col.split("-")[-1] for col in adt_df.columns]

pro_adata = sc.AnnData(adt_df, obs=adata_RNA.obs)
pro_adata.layers["counts"] = pro_adata.X.copy()
pro_adata.obsm["X_umap"] = adata.obsm["X_umap"]
pro_adata.obsm["MultiVI_latent"] = adata.obsm["MultiVI_latent"]
pro_adata.obs["celltype"] = adata.obs["celltype"]

pt.pp.clr(pro_adata)

In [ ]:
marker_proteins = ['CD11b', 'CD36', 'CD18', 'CD55', 'CD35',             # Monocyte
    'CCR2', 'FasL', 'CSF1R', 'TREM1',                                   # MDSC
    'CD276', 'Galectin9', 'PD1', 'CD64', 'CD204', 'MERTK',              # TAM TREM2
    'CD206', 'CD163', 'CD47', 'CXCR4',                                  # TAM FOLR2
    'DNAM1',                                                            # cDC1
    'CD1c', 'SIRPa',                                                    # cDC2
    'CD80', 'CD86', 'PDL1', 'PDL2', 'CD40', 'HLADR', 'CD83', 'IL6Ra',   # mregDC
    'ITGAE',                                                            # preDC
    'CD123', 'BDCA2',                                                   # pDC
    'ckit', 'LAMP1', 'IgE', 'CD9', 'ENPP3'                              # Mast
]

In [ ]:
# Fig.S5A
sc.pl.matrixplot(pro_adata, 
                 marker_proteins, 
                 groupby = "celltype",
                 dendrogram=False,
                 standard_scale='var',                 
                 cmap='plasma',
                 show = False)
plt.savefig(f"{fig_dir}FigureS5A_protein_heatmap.pdf", bbox_inches='tight')
plt.close()

### Concordance corrplot

In [ ]:
tmp_path = "Reproducibility/Data/UC_DOGMA_myeloid_celltype_annotation_RNA_ATAC.txt"
annotation_rna_atac = pd.read_csv(tmp_path, index_col=0, sep='\t')

tmp_path = "Reproducibility/Data/UC_DOGMA_myeloid_celltype_annotation_ADT.txt"
annotation_adt = pd.read_csv(tmp_path, index_col=0, sep='\t')

In [ ]:
def plot_confusion_heatmap(metadata, pred_col, xlabel, ax, title_prefix, celltype_order, vmin=0, vmax=1):
    # accuracy
    accuracy = np.mean(metadata[pred_col].astype(str) == metadata["celltype"].astype(str))
    # confusion matrix normalized by column
    df = metadata.groupby(["celltype", pred_col]).size().unstack(fill_value=0)
    # enforce row/col order
    celltype_order_r = celltype_order[::-1]
    df = df.reindex(index=celltype_order_r, columns=celltype_order, fill_value=0)
    norm_df = df / df.sum(axis=0)
    # plot heatmap on given axis
    heatmap = ax.pcolor(norm_df, cmap="Blues", vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(0.5, len(df.columns), 1))
    ax.set_xticklabels(df.columns, rotation=90)
    ax.set_yticks(np.arange(0.5, len(df.index), 1))
    ax.set_yticklabels(df.index)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Multimodal")
    ax.set_title(f"{title_prefix}\nAccuracy: {accuracy:.2f}")
    return heatmap

In [ ]:
celltype_order = ["Mono","MDSC-like","TAM_TREM2","TAM_FOLR2","cDC1","cDC2",
                  "mregDC","preDC","pDC","Mast"]
metadata_all = pd.read_csv("Reproducibility/Data/UC_DOGMA_metadata.txt", index_col=0, sep='\t')
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

heatmap1 = plot_confusion_heatmap(
    metadata_all.loc[annotation_rna_atac.index].assign(celltype_pvi=annotation_rna_atac["celltype_pvi"]),
    "celltype_pvi", "ATAC", axes[0], "Multimodal vs ATAC", celltype_order
)

heatmap2 = plot_confusion_heatmap(
    metadata_all.loc[annotation_rna_atac.index].assign(celltype_scvi=annotation_rna_atac["celltype_scvi"]),
    "celltype_scvi", "RNA", axes[1], "Multimodal vs RNA", celltype_order
)

heatmap3 = plot_confusion_heatmap(
    metadata_all.loc[annotation_adt.index].assign(celltype_adt=annotation_adt["celltype_adt"]),
    "celltype_adt", "ADT", axes[2], "Multimodal vs ADT", celltype_order
)

cbar_ax = fig.add_axes([0.92, 0.15, 0.01, 0.7])
fig.colorbar(heatmap1, cax=cbar_ax)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.savefig(f"{fig_dir}FigureS5B.pdf", bbox_inches='tight')
plt.close()